# Decision Tree

#### Zahra Jiriaei

In [1]:
# import library
import numpy as np
import pandas as pd
from collections import Counter

Two implementation approaches are presented below. Although the two approaches differ in their implementation methods, they follow the same overall code structure.

The first approach relies primarily on the Pandas library and datasets, whereas the second approach makes greater use of the NumPy library and arrays.

## Decision tree (Method 1)

First, we define a class representing a node. These nodes correspond to the nodes in the decision tree.

The class defines the features, the left and right child nodes, and the value used by the tree to split the data into two subsets. If the node is a leaf node, its corresponding value and gain are also defined.

These are the essential attributes we need to know about a node.


In [37]:
class Node:
    
    def __init__(self,feature=None,threshold=None,left_node=None,right_node=None,value=None,IG=None):
        self.feature=feature
        self.threshold=threshold
        self.left_node=left_node
        self.right_node=right_node
        self.value=value
        self.IG=IG
        
    def Leave_node(self):
        return self.value is not None

The decision tree implemented using the first approach is presented below. The necessary explanations have been provided within the code.

In [38]:
class DecisionTree:
    
    def __init__(self,att_num=None,depth=None,features=None,mode="antropy"):
        self.att_num=att_num
        self.depth=depth
        self.features=features
        self.mode=mode
        self.root=None
        
    def Fit(self,x_train,y_train):
        """
        This function fit tree on train dataset
    
        Parameters:
            x_dataset(dataset): Dataset
            y_train(dataset): Target column

        """
        n_data,n_features=x_train.shape
        if n_features>=self.features:
            self.root=self.GrowTree(x_train,y_train,0) 
              
        else:
            raise ValueError("Change number of the features. That is too much!")
            

    def GrowTree(self,x,y_train,depth):
        """
        This function build tree on train dataset
    
        Parameters:
            x(dataset): Dataset
            y_train(dataset): Target column
            depth(int):depth of the tree
            
        return:
            Node: decision tree node with defined attributes
        """ 
        #Check depth
        if depth<self.depth :
            leaf_value=self.CalculateLeafValue(y_train)
            return Node(value=leaf_value)
            
        best_IG,best_feature_index=self.FindBestFeature(x,y_train)
        left_node,right_node,threshold=self.Split(x,best_feature_index)
        
        #recursive
            
        left_tree=self.GrowTree(left_node,y_train,depth+1)
        right_tree=self.GrowTree(right_node,y_train,depth+1)
        
        return Node(self,feature=best_feature_index,threshold=threshold,left_node=left_node,
                    right_node=right_node,value=None,IG=best_IG)
        

        
    def FindBestFeature(self,x_dataset,y_train):
        
        """
        This function calulate Information Gain
    
        Parameters:
            x_dataset(dataset): Database
            y_train(dataset): Target column

        return:
            IG (float)
            feature_index (int)
        """
        
        IG=-float("inf")
        feature_index=None
        n_data,n_features=x_dataset.shape
        
        for feature_ind in range(0,n_features):
           
            feature_IG=self.IGCalculate(x_dataset,feature_ind,y_train)
            if feature_IG>=IG:
                IG=feature_IG
                feature_index=feature_ind
      
        return IG,feature_index
          
    def IGCalculate(self,x_dataset,feature_index,y_train):
        
        """
        This function calulate Information Gain
    
        Parameters:
            x_dataset(dataset): Database
            feature_index(int): Index of the feature
            y_train(dataset): Target column
        return:
            float  
        """
        
        if self.mode=="antropy":
            # parent antropy
            parent_IG=self.Antropy(y_train,0)
            
            # Make children
            left_children, right_children,threshold=self.Split(x_dataset,feature_index)
            
            # Children antropy
            ## y_index=x_dataset.columns.get_loc(y_train.columns[0])
            left_children_antropy=self.Antropy(left_children,feature_index)
            right_children_antropy=self.Antropy(right_children,feature_index)
            
            # Calculate IG
            weight_left=len(left_children)/len(x_dataset)
            weight_right=len(right_children)/len(x_dataset)
            IG=parent_IG-((weight_left*left_children_antropy)+(weight_right*right_children_antropy))
            return IG
       
        elif self.mode=="gini":
            # parent antropy
            parent_IG=self.Gini(y_train,0)
            
            # Make children
            left_children, right_children=self.Split(x_dataset,feature_index)
            
            # Children antropy
            ## y_index=x_dataset.columns.get_loc(y_train.columns[0])
            left_children_gini=Gini(left_children,feature_index)
            right_children_gini=Gini(right_children,feature_index)
            
            # Calculate IG
            weight_left=len(left_children)/len(x_dataset)
            weight_right=len(right_children)/len(x_dataset)
            IG=parent_IG-((weight_left*left_children_gini)+(weight_right*right_children_gini))
            return IG
        
    def Gini(self,x_dataset,feature_index):
        """
        This function calulate attributes gini index
    
        Parameters:
            x_dataset(dataset): Database
            feature_index(int): index of the feature
        
        return:
            float  
        """
        xi=x_dataset.value_counts()
        xil=xi.tolist()
        xil=np.array(xil)
        pxi=xil/len(x_dataset)
        gini_index=sum(pxi*(1-pxi))
    
        return gini_index
    
    def Antropy(self,x_dataset,feature_index):
        
        """
        This function calulate attributes antropy
    
        Parameters:
            x_dataset(dataset): Database
            feature_index(int): index of the feature
        
        return:
            float  
        """
        #print(type(x_dataset))
        #x_dataset=x_dataset.to_frame()
        #print(type(x_dataset))
        #xi=x_dataset.groupby((x_dataset.columns[feature_index])).size()
        xi=x_dataset.value_counts()
        xil=xi.tolist()
        xil=np.array(xil)
        pxi=xil/len(x_dataset)
        antropy=-1*sum(pxi*np.log2(pxi))
        
        return antropy
    
    def Split(self,x_dataset,feature_index):
        
        """
        This function split dataset to two dataset base on unique value
    
        Parameters:
            x_dataset(dataset): Database
            feature_index(int): index of the feature
        
        return:
            Two dataset as left and right children and treshold(seperator)  
        """
        
        treshold=x_dataset.iloc[ : ,feature_index].drop_duplicates()
        treshold= treshold.values.tolist()
        left_child=x_dataset.groupby((x_dataset.columns[feature_index])).get_group(treshold[0])
        right_child=x_dataset.groupby((x_dataset.columns[feature_index])).get_group(treshold[1])
        
        return left_child,right_child,treshold
        
    def CalculateLeafValue(self,y):
        """
        This function return leaf value
    
        Parameters:
            y(list):list of depthest value in tree
        
        return:
            Nodes value
        """
        counter=Counter(y)
        value= counter.most_common(1)[0][0]
        return  value
    
    def predict(self,X):
        return np.array([self.predicttree(x,self.root) for x in X])
    
    def predicttree(self, x,node):
         """
        This function predict values
    
        Parameters:
            x(dataset)
            node=Tree nodes
        
        return:
            predicted value base on nodes
        """
        if node.Leave_node():
            return node.value
        
        if x[node.feature]<=node.threshold:
            return self.predicttree(x,node.left_node)
        else:
            return self.predicttree(x,node.right_node)
        
    def show_tree(self,node=None):
         """
        This function show fitted tree
    
        Parameters:
            node=Tree nodes
        
        return:
            print nodes and IG of each
        """
        #print("I am running")
        if not node:
            node=self.root
        
        print("feature index:",node.feature,"IG:",node.IG) 
        self.show_tree(node.left_node)
        print("feature index:",node.feature,"IG:",node.IG) 
        self.show_tree(node.right_node)

## Fit tree on restaurent dataset

In [39]:
# import data
resdb=pd.read_excel("E:\IUST\year 4\8\AL\Code\Decision tree\Restaurent.xlsx") 

In [40]:
resdb.iloc[:,1:-1]

,Alt,Bar,Fri,Hun,Pat,Price,Rain,Res,Type,Est
0,yes,no,no,yes,some,3,No,yes,french,0-10
1,yes,no,no,yes,full,1,No,no,thai,30-60
2,no,yes,no,no,some,1,No,no,burger,0-10
3,yes,no,yes,yes,full,1,yes,no,thai,10-30
4,yes,no,yes,no,full,3,no,yes,french,60<
5,no,yes,no,yes,some,2,yes,yes,italian,0-10
6,no,yes,no,no,none,1,yes,no,burger,0-10
7,no,no,no,yes,some,2,yes,yes,thai,0-10
8,no,yes,yes,no,full,1,yes,no,burger,60<
9,yes,yes,yes,yes,full,3,no,yes,italian,10-30


In [ ]:
# Prepare the dataset

X=resdb.iloc[:,1:-1]
Y=resdb.iloc[:,-1]

# Split the data into training and test sets

from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.3,random_state=52)

# Fit the model and construct the decision tree using the dataset

instance1=DecisionTree(depth=20,features=5,mode="gini")
instance1.Fit(X_train,Y_train)

# Make predictions

predict=instance1.predict(X_test)
current=np.array(Y_train.tolist()[0:len(predict)])
predict=instance1.predict(X_test)[0:len(current)]

# Calculate accuracy

from sklearn.metrics import accuracy_score
accuracy_score(current,predict)

0.75

In [ ]:
# Prepare the dataset

X=resdb.iloc[:,1:-1]
Y=resdb.iloc[:,-1]

# Split the data into training and test sets

from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.4,random_state=52)

# Fit the model and construct the decision tree using the dataset

instance1=DecisionTree(depth=20,features=5,mode="gini")
instance1.Fit(X_train,Y_train)

# Make predictions

predict=instance1.predict(X_test)
current=np.array(Y_train.tolist()[0:len(predict)])
predict=instance1.predict(X_test)[0:len(current)]

# Calculate accuracy

from sklearn.metrics import accuracy_score
accuracy_score(current,predict)

0.7142857142857143

In [ ]:
# Prepare the dataset

X=resdb.iloc[:,1:-1]
Y=resdb.iloc[:,-1]

# Split the data into training and test sets

from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.5,random_state=52)

# Fit the model and construct the decision tree using the dataset

instance1=DecisionTree(depth=20,features=5,mode="gini")
instance1.Fit(X_train,Y_train)

# Make predictions

predict=instance1.predict(X_test)
current=np.array(Y_train.tolist()[0:len(predict)])
predict=instance1.predict(X_test)[0:len(current)]

# Calculate accuracy

from sklearn.metrics import accuracy_score
accuracy_score(current,predict)

0.6666666666666666

##### As can be observed, increasing the proportion of the test set resulted in a decrease in accuracy in this case.


In [ ]:
# Prepare the dataset

X=resdb.iloc[:,1:-1]
Y=resdb.iloc[:,-1]

# Split the data into training and test sets

from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.3,random_state=52)

# Fit the model and construct the decision tree using the dataset

instance1=DecisionTree(depth=20,features=5,mode="antropy")
instance1.Fit(X_train,Y_train)

# Make predictions

predict=instance1.predict(X_test)
current=np.array(Y_train.tolist()[0:len(predict)])
predict=instance1.predict(X_test)[0:len(current)]

# Calculate accuracy

from sklearn.metrics import accuracy_score
accuracy_score(current,predict)

0.75

In [ ]:
# Prepare the dataset

X=resdb.iloc[:,1:-1]
Y=resdb.iloc[:,-1]

# Split the data into training and test sets

from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.4,random_state=52)

# Fit the model and construct the decision tree using the dataset

instance1=DecisionTree(depth=20,features=5,mode="antropy")
instance1.Fit(X_train,Y_train)

# Make predictions

predict=instance1.predict(X_test)
current=np.array(Y_train.tolist()[0:len(predict)])
predict=instance1.predict(X_test)[0:len(current)]

# Calculate accuracy

from sklearn.metrics import accuracy_score
accuracy_score(current,predict)

0.7142857142857143

In [ ]:
# Prepare the dataset

X=resdb.iloc[:,1:-1]
Y=resdb.iloc[:,-1]

# Split the data into training and test sets

from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.5,random_state=52)

# Fit the model and construct the decision tree using the dataset

instance1=DecisionTree(depth=20,features=5,mode="antropy")
instance1.Fit(X_train,Y_train)

# Make predictions

predict=instance1.predict(X_test)
current=np.array(Y_train.tolist()[0:len(predict)])
predict=instance1.predict(X_test)[0:len(current)]

# Calculate accuracy

from sklearn.metrics import accuracy_score
accuracy_score(current,predict)

0.6666666666666666

##### As can be observed, there is no significant difference in accuracy between the Gini index and entropy.


### Decision Tree Presentation

The following function was used to visualize the constructed decision tree. However, although the function appears to be correctly implemented, the code in this section does not run.


In [11]:
#instance1.show_tree()

## Fit tree on taitanic dataset

In [12]:
taitanic=pd.read_csv("E:/IUST/year 4/8/AL/Code/Decision tree/titanic.csv") 

In [13]:
taitanic.head(10)

,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,survived
0,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,1
1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,1
2,1,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,0
3,1,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,0
4,1,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,0
5,1,"Anderson, Mr. Harry",male,48.0000,0,0,19952,26.5500,E12,S,1
6,1,"Andrews, Miss. Kornelia Theodosia",female,63.0000,1,0,13502,77.9583,D7,S,1
7,1,"Andrews, Mr. Thomas Jr",male,39.0000,0,0,112050,0.0000,A36,S,0
8,1,"Appleton, Mrs. Edward Dale (Charlotte Lamson)",female,53.0000,2,0,11769,51.4792,C101,S,1
9,1,"Artagaveytia, Mr. Ramon",male,71.0000,0,0,PC 17609,49.5042,NaN,C,0


In [ ]:
# Prepare the dataset and remove missing values
taitanic.dropna(inplace=True)

# Remove the column that is not needed
X=taitanic.drop("name",axis=1,inplace=True)

In [ ]:
# Prepare the dataset
X=taitanic.iloc[:,0:-1]
Y=taitanic.iloc[:,-1]

# Split the dataset into training and test sets
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.3,random_state=52)

# Construct the decision tree
instance2=DecisionTree(depth=20,features=5,mode="gini")
instance2.Fit(X_train,Y_train)

# Make predictions
predict=instance2.predict(X_test)
current=np.array(Y_train.tolist()[0:len(predict)])

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy_score(current,predict)

0.6666666666666666

In [ ]:
# Prepare the dataset
X=taitanic.iloc[:,0:-1]
Y=taitanic.iloc[:,-1]

# Split the dataset into training and test sets
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.4,random_state=52)

# Construct the decision tree
instance2=DecisionTree(depth=20,features=5,mode="gini")
instance2.Fit(X_train,Y_train)

# Make predictions
predict=instance2.predict(X_test)
current=np.array(Y_train.tolist()[0:len(predict)])

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy_score(current,predict)

0.7777777777777778

In [ ]:
# Prepare the dataset
X=taitanic.iloc[:,0:-1]
Y=taitanic.iloc[:,-1]

# Split the dataset into training and test sets
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.5,random_state=52)

# Construct the decision tree
instance2=DecisionTree(depth=20,features=5,mode="gini")
instance2.Fit(X_train,Y_train)

# Make predictions
predict=instance2.predict(X_test)
current=np.array(Y_train.tolist()[0:len(predict)])

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy_score(current,predict)

0.5555555555555556

##### As can be observed, the highest accuracy was achieved when 40% of the data was used as the test set.

##### Therefore, it can be concluded that the proportion of data allocated to the test set does not appear to have a strong relationship with accuracy.


In [ ]:
# Prepare the dataset
X=taitanic.iloc[:,0:-1]
Y=taitanic.iloc[:,-1]

# Split the dataset into training and test sets
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.5,random_state=52)

# Construct the decision tree
instance2=DecisionTree(depth=20,features=5,mode="antropy")
instance2.Fit(X_train,Y_train)

# Make predictions
predict=instance2.predict(X_test)
current=np.array(Y_train.tolist()[0:len(predict)])

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy_score(current,predict)

0.5555555555555556

In [ ]:
# Prepare the dataset
X=taitanic.iloc[:,0:-1]
Y=taitanic.iloc[:,-1]

# Split the dataset into training and test sets
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.4,random_state=52)

# Construct the decision tree
instance2=DecisionTree(depth=20,features=5,mode="antropy")
instance2.Fit(X_train,Y_train)

# Make predictions
predict=instance2.predict(X_test)
current=np.array(Y_train.tolist()[0:len(predict)])

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy_score(current,predict)

0.7777777777777778

In [ ]:
# Prepare the dataset
X=taitanic.iloc[:,0:-1]
Y=taitanic.iloc[:,-1]

# Split the dataset into training and test sets
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.5,random_state=52)

# Construct the decision tree
instance2=DecisionTree(depth=20,features=5,mode="antropy")
instance2.Fit(X_train,Y_train)

# Make predictions
predict=instance2.predict(X_test)
current=np.array(Y_train.tolist()[0:len(predict)])

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy_score(current,predict)

0.5555555555555556

##### As can be observed, the results obtained using the two criteria differ slightly. On average, the Gini index performs better in this example. However, based on the results for both criteria, higher accuracy was achieved when 40% of the data was used as the test set.

### Decision Tree Presentation

In [70]:
#instance1.show_tree()

## Decision tree (Method 2)

In [20]:
class Node():

    def __init__(self,feature=None,threshold=None,left_node=None,right_node=None,IG=None,value="ThisisNone"):
        self.feature=feature
        self.threshold=threshold
        self.left_node=left_node
        self.right_node=right_node
        self.value=value
        self.IG=IG

In [75]:
class DecisionTree():
    
    def __init__(self,sample_min=None,depth=None):
        self.root=None
        self.sample_min=sample_min
        self.depth=depth
        
    def GrowTree(self,dataset,depth=0):
        """
        This function build tree on train dataset
    
        Parameters:
            dataset(dataset)
            depth(int): It shows depth of the tree
        
        return:
            decision tree node with defined info
        """
        x,y=dataset[:,:-1],dataset[:,-1]
        samples_num,feature_num=np.shape(x)
        
        if samples_num>=self.sample_min and depth<=self.depth:
            best_split=self.BestSplit(dataset,samples_num,feature_num)
            #print(best_split)
            if best_split["info_gain"]>0:
                # recursive
                left_subtree=self.GrowTree(best_split["dataset_left"],depth+1)
                right_subtree=self.GrowTree(best_split["dataset_right"],depth+1)
                
                return Node(best_split["feature_index"],best_split["threshold"],
                            left_subtree,right_subtree,best_split["info_gain"])
            
        leaf_value = self.calculate_leafe_value(y)
        return Node(value=leaf_value)
    
    def BestSplit(self,dataset,num_samples,num_features):
        """
        This function make nodes and its children
    
        Parameters:
            dataset(dataset)
            num_samples(int): number of dataset rows
            num_features(int); number of featur in node
        
        return:
            node
        """
        best_split={}
        max_info_gain=-float("inf")
                
        for feature_index in range(num_features):
            feature_value=dataset[:,feature_index]
            possible_threshold=np.unique(feature_value)
            
            for threshold in possible_threshold:
                detaset_left,dataset_right=self.split(dataset,feature_index,threshold)
                
                if len(detaset_left)>0 and len(dataset_right)>0:
                    y,left_y,right_y=dataset[:,-1],detaset_left[:,-1],dataset_right[:,-1]
                    curr_info_gain=self.information_gain(y, left_y,right_y,"gini")
                    
                    if curr_info_gain>max_info_gain:
                        best_split["feature_index"]=feature_index
                        best_split["threshold"]=threshold
                        best_split["dataset_left"]=detaset_left
                        best_split["dataset_right"]=dataset_right
                        best_split["info_gain"]=curr_info_gain
        return best_split
    
    def split(self,dataset,feature_index,threshold):
        """
        This function split dataset in two
    
        Parameters:
            dataset(dataset)
            feature_index(int): we split base on feature_index
            threshold(str): value that split node
        
        return:
            righ and left dataset as right and left children
        """
        feature_value=dataset[:,feature_index]
        possible_threshold=np.unique(feature_value)
        other_threshold=np.delete(possible_threshold,np.where(possible_threshold == threshold))
        # print(feature_index,threshold)
        dataset_left=np.array([row for row in dataset if (row[threshold in possible_threshold]).any()])        
        dataset_right=np.array([row for row in dataset if (row[threshold not in other_threshold]).any()])
        return dataset_left,dataset_right
       
    def information_gain(self,parent,l_child,r_child,mode="entropy"):
        
        """
        This function calulate Information Gain
    
        Parameters:
            parent(dataset)
            l_child(dataset)
            r_child(dataset)
            mode="entropy"
        return:
            float  
        """
        
        weight_l=len(l_child)/len(parent)
        weight_r=len(r_child)/len(parent)
        
        if mode=="gini":
            gain=self.gini_index(parent)-(weight_l*self.gini_index(l_child)+weight_r*self.gini_index(r_child))
        else:
            gain=self.entropy(parent)-(weight_l*self.entropy(l_child)+weight_r*self.entropy(r_child))
        return gain
    
    def entropy(self,y):
        """
        This function calulate attributes antropy
    
        Parameters:
            y(dataset): Dataset

        
        return:
            float  
        """
        class_labls=np.unique(y)
        entropy=0
        for cls in class_lables:
            p_cls=len(y[y==cls])/len(y)
            entropy += -p_cls *np.log2(p_cls)
            
        return entropy
    
    def gini_index(self,y):
        """
        This function calulate attributes gini index
    
        Parameters:
            y(dataset): Dataset

        
        return:
            float  
        """
        class_labls=np.unique(y)
        gini=0
        for cls in class_labls:
            p_cls=len(y[y==cls])/len(y)
            gini += -p_cls **2
            
        return 1-gini
    
    def calculate_leafe_value(self,y):
        """
        This function return leaf value
    
        Parameters:
            y(list):list of depthest value in tree
        
        return:
            Nodes value
        """
        y=list(y)
        return max(y,key=y.count)
    
    def fit(self,x,y):
        """
        This function fit tree on dataset
    
        Parameters:
            x(dataset): X train
            y(dataset): Y train
        
        """
        dataset=np.concatenate((x,y),axis=1)
        self.root=self.GrowTree(dataset)
    
    def make_predict(self,x,tree):
        
        if tree.value!="ThisisNone":
            return tree.value
        feature_val=x[tree.feature_index]
        
        if feature_val<=tree.threshold:
            return self.make_predict(x,tree.left)
        else:
            return self.make_predict(x,tree.right)
        
    def predict(self,X):
        """
        This function predict value with make_predict function
    
        Parameters:
            x(dataset): X test
            
        return:
            predicted value
        """
        prediction=[self.make_predict(x,self.root) for x in X]
        return prediction
    
    def show_tree(self,node=None):
        """
        This function show fitted tree
    
        Parameters:
            node=Tree nodes
        
        return:
            print nodes and IG of each
        """
        #print("I am running")
        if not node:
            node=self.root
        
        print("feature index:",node.feature,"IG:",node.IG) 
        self.show_tree(node.left_node)
        print("feature index:",node.feature,"IG:",node.IG) 
        self.show_tree(node.right_node)
    

## Fit tree on Resturant dataset


In [76]:
# import data
resdb=pd.read_excel("E:\IUST\year 4\8\AL\Code\Decision tree\Restaurent.xlsx")

In [ ]:
# Prepare the data
X=resdb.iloc[:,:-1].values
Y=resdb.iloc[:,-1].values.reshape(-1,1)

# Split the data into training and test sets
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.5,random_state=52)

# Fit the decision tree
instance4=DecisionTree(5,5)
instance4.fit(X_train,Y_train)

# Make predictions
predict=instance4.predict(X_test)
current=np.array(Y_train.tolist()[0:len(predict)])

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy_score(current,predict)

0.6666666666666666

In [ ]:
# Prepare the data
X=resdb.iloc[:,:-1].values
Y=resdb.iloc[:,-1].values.reshape(-1,1)

# Split the data into training and test sets
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.4,random_state=52)

# Fit the decision tree
instance4=DecisionTree(5,5)
instance4.fit(X_train,Y_train)

# Make predictions
predict=instance4.predict(X_test)
current=np.array(Y_train.tolist()[0:len(predict)])

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy_score(current,predict)

0.6

In [ ]:
# Prepare the data
X=resdb.iloc[:,:-1].values
Y=resdb.iloc[:,-1].values.reshape(-1,1)

# Split the data into training and test sets
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.3,random_state=52)

# Fit the decision tree
instance4=DecisionTree(5,5)
instance4.fit(X_train,Y_train)

# Make predictions
predict=instance4.predict(X_test)
current=np.array(Y_train.tolist()[0:len(predict)])

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy_score(current,predict)

1.0

##### In this example, the highest accuracy was achieved when 30% of the data was used as the test set, which is considerably better than the results obtained with the previously implemented class.

In [ ]:
# Prepare the data
X=resdb.iloc[:,:-1].values
Y=resdb.iloc[:,-1].values.reshape(-1,1)

# Split the data into training and test sets
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.5,random_state=52)

# Fit the decision tree
instance4=DecisionTree(5,5)
instance4.fit(X_train,Y_train)

# Make predictions
predict=instance4.predict(X_test)
current=np.array(Y_train.tolist()[0:len(predict)])

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy_score(current,predict)

0.6666666666666666

In [ ]:
# Prepare the data
X=resdb.iloc[:,:-1].values
Y=resdb.iloc[:,-1].values.reshape(-1,1)

# Split the data into training and test sets
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.5,random_state=52)

# Fit the decision tree
instance4=DecisionTree(3,5)
instance4.fit(X_train,Y_train)

# Make predictions
predict=instance4.predict(X_test)
current=np.array(Y_train.tolist()[0:len(predict)])

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy_score(current,predict)

0.6666666666666666

In [ ]:
# Prepare the data
X=resdb.iloc[:,:-1].values
Y=resdb.iloc[:,-1].values.reshape(-1,1)

# Split the data into training and test sets
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.5,random_state=52)

# Fit the decision tree
instance4=DecisionTree(5,3)
instance4.fit(X_train,Y_train)

# Make predictions
predict=instance4.predict(X_test)
current=np.array(Y_train.tolist()[0:len(predict)])

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy_score(current,predict)

0.6666666666666666

##### As can be observed, changing other parameters of the decision tree, such as its depth, does not result in any noticeable change in accuracy in this example.

### Desicion tree presentation

در این جا هم مثل قبل خروجی ندارد

In [29]:
#instance4.show_tree()

## Fit tree on taitanic dataset


In [30]:
taitanic=pd.read_csv("E:/IUST/year 4/8/AL/Code/Decision tree/titanic.csv") 

In [ ]:
taitanic.dropna(inplace=True)
X=taitanic.drop("name",axis=1,inplace=True)

In [ ]:
# Prepare the dataset
X=taitanic.iloc[:,:-1].values
Y=taitanic.iloc[:,-1].values.reshape(-1,1)

# Split the dataset into training and test sets
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=41)

# Fit the model and construct the decision tree
instance3=DecisionTree(3,3)
instance3.fit(X_train,Y_train)

# Make predictions
predict=instance3.predict(X_test)

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy_score(Y_test,predict)

0.5

In [ ]:
# Prepare the dataset
X=taitanic.iloc[:,:-1].values
Y=taitanic.iloc[:,-1].values.reshape(-1,1)

# Split the dataset into training and test sets
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.3,random_state=41)

# Fit the model and construct the decision tree
instance3=DecisionTree(3,3)
instance3.fit(X_train,Y_train)

# Make predictions
predict=instance3.predict(X_test)

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy_score(Y_test,predict)

0.5555555555555556

In [ ]:
# Prepare the dataset
X=taitanic.iloc[:,:-1].values
Y=taitanic.iloc[:,-1].values.reshape(-1,1)

# Split the dataset into training and test sets
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.4,random_state=41)

# Fit the model and construct the decision tree
instance3=DecisionTree(3,3)
instance3.fit(X_train,Y_train)

# Make predictions
predict=instance3.predict(X_test)

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy_score(Y_test,predict)

0.5740740740740741

##### In this example, increasing the proportion of test data resulted in higher accuracy.

##### However, as concluded earlier, the proportion of test data does not appear to be correlated with accuracy.


In [ ]:
# Prepare the dataset
X=taitanic.iloc[:,:-1].values
Y=taitanic.iloc[:,-1].values.reshape(-1,1)

# Split the dataset into training and test sets
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.4,random_state=41)

# Fit the model and construct the decision tree
instance3=DecisionTree(100,3)
instance3.fit(X_train,Y_train)

# Make predictions
predict=instance3.predict(X_test)

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy_score(Y_test,predict)

0.5740740740740741

In [ ]:
# Prepare the dataset
X=taitanic.iloc[:,:-1].values
Y=taitanic.iloc[:,-1].values.reshape(-1,1)

# Split the dataset into training and test sets
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.4,random_state=41)

# Fit the model and construct the decision tree
instance3=DecisionTree(100,10)
instance3.fit(X_train,Y_train)

# Make predictions
predict=instance3.predict(X_test)

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy_score(Y_test,predict)

0.5740740740740741

##### As can be observed, changing other parameters of the decision tree, such as its depth, does not result in any noticeable change in accuracy in this example.
##### However, based on the literature and previous findings in this area, increasing the tree depth would be expected to affect accuracy, while using a larger amount of training data would generally be expected to improve the model's accuracy.
